In [ ]:

from datetime import datetime
from prettytable import PrettyTable

# Google Pay Expense Sharing Application

friends = ["Vikram", "Siva", "Naren"]
expenses = []


# Add a new friend
def add_friend(name):
    name = name.strip().title()

    if not name:
        print("Please enter a valid name!")
    elif name in friends:
        print("Friend already exists!")
    else:
        friends.append(name)
        print(name, "added successfully!")


# Add an expense with equal splitting
def add_equal_expense(description, payer, amount, beneficiaries):
    payer = payer.strip().title()
    beneficiaries = [name.strip().title() for name in beneficiaries]

    if payer not in friends:
        print("Payer not found!")
        return

    if not beneficiaries or any(name not in friends for name in beneficiaries):
        print("Please enter valid beneficiaries!")
        return

    if len(set(beneficiaries)) != len(beneficiaries):
        print("Duplicate beneficiaries are not allowed!")
        return

    if amount <= 0:
        print("Amount must be greater than zero!")
        return

    share = round(amount / len(beneficiaries), 2)
    shares = {name: share for name in beneficiaries}

    # Adjust the last share to handle rounding
    shares[beneficiaries[-1]] = round(
        amount - sum(shares[name] for name in beneficiaries[:-1]), 2
    )

    expenses.append({
        "description": description,
        "payer": payer,
        "amount": round(amount, 2),
        "shares": shares,
        "date": datetime.now().strftime("%d-%m-%Y %H:%M")
    })

    print("Expense added successfully!")


# Add an expense with custom splitting
def add_custom_expense(description, payer, amount, shares):
    payer = payer.strip().title()
    shares = {name.strip().title(): value for name, value in shares.items()}

    if payer not in friends:
        print("Payer not found!")
        return

    if not shares or any(name not in friends for name in shares):
        print("Please enter valid friends!")
        return

    if amount <= 0 or any(value < 0 for value in shares.values()):
        print("Enter a valid amount and shares!")
        return

    if round(sum(shares.values()), 2) != round(amount, 2):
        print("Total shares must equal the expense amount!")
        return

    expenses.append({
        "description": description,
        "payer": payer,
        "amount": round(amount, 2),
        "shares": shares.copy(),
        "date": datetime.now().strftime("%d-%m-%Y %H:%M")
    })

    print("Custom expense added successfully!")


# Display transaction history
def show_expenses():
    if not expenses:
        print("No expenses recorded yet!")
        return

    table = PrettyTable()
    table.field_names = [
        "Description", "Paid By", "Amount", "Split Details", "Date"
    ]

    for expense in expenses:
        table.add_row([
            expense["description"],
            expense["payer"],
            f"₹{expense['amount']:.2f}",
            str(expense["shares"]),
            expense["date"]
        ])

    print(table)


# Calculate everyone's balance
def calculate_balances():
    balances = {friend: 0.0 for friend in friends}

    for expense in expenses:
        balances[expense["payer"]] += expense["amount"]

        for friend, share in expense["shares"].items():
            balances[friend] -= share

    return {
        friend: round(balance, 2)
        for friend, balance in balances.items()
    }


# Display balance summary
def show_balances():
    balances = calculate_balances()
    table = PrettyTable()
    table.field_names = ["Friend", "Settlement"]

    for friend, balance in balances.items():
        if balance > 0.01:
            message = f"Should receive ₹{balance:.2f}"
        elif balance < -0.01:
            message = f"Owes ₹{-balance:.2f}"
        else:
            message = "Settled"

        table.add_row([friend, message])

    print(table)


# Suggest payments to settle debts
def suggest_payments():
    balances = calculate_balances()

    creditors = [
        [name, amount]
        for name, amount in balances.items()
        if amount > 0.01
    ]

    debtors = [
        [name, -amount]
        for name, amount in balances.items()
        if amount < -0.01
    ]

    table = PrettyTable()
    table.field_names = ["Who Pays", "Who Receives", "Amount"]

    while debtors and creditors:
        debtor, debt_amount = debtors.pop(0)
        creditor, credit_amount = creditors.pop(0)

        payment = min(debt_amount, credit_amount)

        table.add_row([debtor, creditor, f"₹{payment:.2f}"])

        debt_amount = round(debt_amount - payment, 2)
        credit_amount = round(credit_amount - payment, 2)

        if debt_amount > 0.01:
            debtors.insert(0, [debtor, debt_amount])

        if credit_amount > 0.01:
            creditors.insert(0, [creditor, credit_amount])

    print("\nSuggested Payments:")
    print(table if table.rowcount else "Everyone is settled!")


# Main application menu
def expense_sharing_app():
    while True:
        print("\n===== GOOGLE PAY EXPENSE SHARING =====")
        print("1. View Friends")
        print("2. Add Friend")
        print("3. Add Equal Expense")
        print("4. Add Custom Expense")
        print("5. View Transaction History")
        print("6. View Balances")
        print("7. Suggest Payments")
        print("8. Exit")

        choice = input("Enter your choice: ").strip()

        try:
            if choice == "1":
                print("Friends:", ", ".join(friends))

            elif choice == "2":
                add_friend(input("Enter friend name: "))

            elif choice == "3":
                description = input("Expense description: ")
                payer = input("Who paid? ")
                amount = float(input("Enter total amount: ₹"))

                print("Available friends:", ", ".join(friends))
                names = input(
                    "Enter beneficiaries separated by commas: "
                )
                beneficiaries = [
                    name.strip() for name in names.split(",")
                    if name.strip()
                ]

                add_equal_expense(
                    description, payer, amount, beneficiaries
                )

            elif choice == "4":
                description = input("Expense description: ")
                payer = input("Who paid? ")
                amount = float(input("Enter total amount: ₹"))

                shares = {}
                print("Enter each friend's share.")
                print("Type 'done' when finished.")

                while True:
                    name = input("Friend name: ").strip()

                    if name.lower() == "done":
                        break

                    name = name.title()

                    if name not in friends or name in shares:
                        print("Invalid or duplicate friend!")
                        continue

                    share = float(input("Enter their share: ₹"))

                    if share < 0:
                        print("Share cannot be negative!")
                        continue

                    shares[name] = round(share, 2)

                add_custom_expense(
                    description, payer, amount, shares
                )

            elif choice == "5":
                show_expenses()

            elif choice == "6":
                show_balances()

            elif choice == "7":
                suggest_payments()

            elif choice == "8":
                print("Thank you for using Expense Sharing!")
                break

            else:
                print("Invalid choice! Please try again.")

        except ValueError:
            print("Please enter a valid number.")


# Start the application
expense_sharing_app()



===== GOOGLE PAY EXPENSE SHARING =====
1. View Friends
2. Add Friend
3. Add Equal Expense
4. Add Custom Expense
5. View Transaction History
6. View Balances
7. Suggest Payments
8. Exit


Enter your choice:  1


Friends: Vikram, Siva, Naren

===== GOOGLE PAY EXPENSE SHARING =====
1. View Friends
2. Add Friend
3. Add Equal Expense
4. Add Custom Expense
5. View Transaction History
6. View Balances
7. Suggest Payments
8. Exit


Enter your choice:  3
Expense description:  food
Who paid?  vikram paid
Enter total amount: ₹ 780


Available friends: Vikram, Siva, Naren


...

Enter your choice:  5


No expenses recorded yet!

===== GOOGLE PAY EXPENSE SHARING =====
1. View Friends
2. Add Friend
3. Add Equal Expense
4. Add Custom Expense
5. View Transaction History
6. View Balances
7. Suggest Payments
8. Exit


Enter your choice:  7



Suggested Payments:
Everyone is settled!

===== GOOGLE PAY EXPENSE SHARING =====
1. View Friends
2. Add Friend
3. Add Equal Expense
4. Add Custom Expense
5. View Transaction History
6. View Balances
7. Suggest Payments
8. Exit


Enter your choice:  3
